In [ ]:
import torch

name = torch.cuda.get_device_name(0)
cc   = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"{name} | sm_{cc[0]}{cc[1]} | {vram:.1f} GiB | torch {torch.__version__}")

if cc[0] < 8:
    raise SystemExit(
        f"\nSTOP: {name} is sm_{cc[0]}{cc[1]}. FreeToken needs sm_86+ "
        "(RTX 30/40/50-class).\nRuntime > Change runtime type > L4 or A100.")
print("\nGPU supported — continue.")

NVIDIA L4 | sm_89 | 22.0 GiB | torch 2.11.0+cu128

GPU supported — continue.


In [ ]:
!pip -q install -U uv
!uv pip install --system "freetoken[accel]"
!uv pip install --system -q "nvidia-cuda-nvrtc>=13,<14" "nvidia-cuda-runtime>=13,<14" "nvidia-cublas>=13,<14"

!ft --version

!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i -E /content/cloudflared.deb > /dev/null 2>&1
!cloudflared --version

import glob
print("\nlibs found:")
for p in sorted(glob.glob("/usr/local/lib/python3.13/dist-packages/nvidia/**/*.so.13*",
                          recursive=True)):
    print("   ", p)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 122.9 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 103 packages in 471ms
Prepared 21 packages in 5.00s
Uninstalled 2 packages in 13ms
Installed 21 packages in 121ms
 + apache-tvm-ffi==0.1.13.post3
 - cuda-core==0.3.2
 + cuda-core==1.2.0
 + cuda-tile==1.6.0
 + flashinfer-python==0.6.18.post1
 + flashlib==0.3.0
 + freetoken==0.1.3
 + gguf==0.19.0
 + modelscope==1.40.1
 + modelscope-hub==0.4.3
 + nccl4py==0.5.0
 + ninja==1.13.2
 + nvidia-cuda-nvdisasm==13.4.92
 + nvidia-cudnn-frontend==1.29.0
 + nvidia-cutlass-dsl==4.7.1
 + nvidia-cutlass-dsl-libs-base==4.7.1
 + nvidia-cutlass-dsl-libs-core==4.7.1
 + nvidia-cutlass-dsl-libs-cu12==4.7.1
 + nvidia-cutlass-dsl-libs-cu13==4.7.1
 + partial-json-parser==0.2.1.1.post7
 - pyzmq==26.2.1
 + pyzmq==27.2.0
 + sglang-kernel==0.4.5
freetoken version 0.1.3
cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)

libs found:
    /usr/local/lib/python3.13/dist-packages/nvidi

In [ ]:
import glob, os, re, sys, subprocess, torch

SITE = "/usr/local/lib/python3.13/dist-packages"
CC   = "".join(map(str, torch.cuda.get_device_capability(0)))

LIBDIRS = sorted({os.path.dirname(p)
                  for p in glob.glob(f"{SITE}/nvidia/**/*.so.13*", recursive=True)})
os.environ["LD_LIBRARY_PATH"] = ":".join(
    LIBDIRS + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":")
print("CUDA 13 lib dirs:", *LIBDIRS, sep="\n   ")

print(f"\nsgl_kernel binaries (need sm{CC} or a top-level one):")
for f in sorted(glob.glob(f"{SITE}/sgl_kernel/**/common_ops*", recursive=True)):
    print("   ", f.replace(SITE + "/", ""))

probe = subprocess.run([sys.executable, "-c", "import sgl_kernel; print('sgl_kernel import OK')"],
                       capture_output=True, text=True, env=os.environ)
out = (probe.stdout + probe.stderr).strip()
print("\n" + out[-1500:])

print("\n" + "=" * 70)

CUDA 13 lib dirs:
   /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib

sgl_kernel binaries (need sm89 or a top-level one):
    sgl_kernel/sm100/common_ops.abi3.so
    sgl_kernel/sm90/common_ops.abi3.so

sgl_kernel import OK



In [ ]:
import signal, time, json, requests

MODEL, SERVED, PORT = "nvidia/Qwen3.6-35B-A3B-NVFP4", "qwen", 1919
BASE, LOG = f"http://127.0.0.1:{PORT}", "/content/freetoken.log"

vram        = torch.cuda.get_device_properties(0).total_memory / 1024**3
MOE_SLOTS   = int(vram * 0.40 * 1024 / 1.6934)   # ~40% of VRAM, lower if it OOMs later
MAX_SEQ     = 16384
MAX_RUNNING = 2

_h = subprocess.run(["ft", "serve", "--help"], capture_output=True, text=True)
SUPPORTED = set(re.findall(r"--[a-zA-Z0-9][a-zA-Z0-9-]*", _h.stdout + _h.stderr))


expert cache: 5329 slots (~8.8 GiB of 22.0 GiB)

ft serve --model nvidia/Qwen3.6-35B-A3B-NVFP4 --host 127.0.0.1 --port 1919 --served-model-name qwen --moe-cache-size 5329 --max-seq-len-override 16384 --kv-reserve-tokens 16384 --max-running-requests 2 



In [ ]:
def _strays():
    out = subprocess.run(["pgrep", "-f", "ft serve"], capture_output=True, text=True).stdout.split()
    return [int(p) for p in out if int(p) not in (os.getpid(), os.getppid())]

def stop_server():
    for pid in _strays():
        try: os.kill(pid, signal.SIGTERM)
        except Exception: pass
    for _ in range(20):
        if not _strays(): break
        time.sleep(0.5)
    for pid in _strays():
        try: os.kill(pid, signal.SIGKILL)
        except Exception: pass

def alive(): return bool(_strays())

def diagnose(n=25):
    try: lines = open(LOG).read().replace("\r", "\n").splitlines()
    except FileNotFoundError: return "no log yet"
    keys = ("OutOfMemory", "Backend worker", "Backend supervisor", "ImportError",
            "Traceback", "Free memory", "moe_cache", "Shutting down")
    hits = [l for l in lines if any(k in l for k in keys)]
    return ("\n".join(hits[-n:]) + "\n--- tail ---\n" + "\n".join(lines[-n:])).strip()

def start_server():
    stop_server()
    cmd = ["ft", "serve", "--model", MODEL, "--host", "127.0.0.1", "--port", str(PORT)]
    def add(flag, val=None, alt=None):
        f = flag if flag in SUPPORTED else (alt if alt in SUPPORTED else None)
        if f is None: print(f"  (skipping {flag} — not in this build)"); return
        cmd.append(f)
        if val is not None: cmd.append(str(val))
    add("--served-model-name", SERVED)
    add("--moe-cache-size", MOE_SLOTS)
    add("--max-seq-len-override", MAX_SEQ)
    add("--kv-reserve-tokens", MAX_SEQ)
    add("--max-running-requests", MAX_RUNNING, alt="--max-running-req")
    print(" ".join(cmd), "\n")
    return subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True, env=os.environ)

print(f"expert cache: {MOE_SLOTS} slots (~{MOE_SLOTS*1.6934/1024:.1f} GiB of {vram:.1f} GiB)\n")
proc = start_server()

In [ ]:
def loaded():
    """A real 1-token completion. /v1/models answers long before the worker is ready."""
    try:
        r = requests.post(f"{BASE}/v1/chat/completions",
                          json={"model": SERVED, "messages": [{"role": "user", "content": "hi"}],
                                "max_tokens": 1, "stream": False}, timeout=30)
        return r.status_code == 200
    except requests.exceptions.RequestException:
        return False

t0, ok = time.time(), False
while time.time() - t0 < 7200:
    if proc.poll() is not None:
        print(f"\nserver exited (code {proc.returncode})\n"); print(diagnose()); break
    if loaded():
        ok = True; print(f"\nLOADED after {time.time()-t0:.0f}s"); break
    try:
        last = [l for l in open(LOG).read().replace("\r", "\n").splitlines() if l.strip()][-1]
    except Exception:
        last = ""
    print(f"  {time.time()-t0:6.0f}s | {last[:100]:<100}", end="\r")
    time.sleep(15)


LOADED after 347s
ServerArgs(model_path='nvidia/Qwen3.6-35B-A3B-NVFP4', tp_info=DistributedInfo(rank=0, size=1), dtype=torch.bfloat16, max_running_req=2, attention_backend='auto', moe_strategy='auto', quant_backend=None, ple_backend='disk', expert_load='auto', moe_cache_size=5329, moe_cache_rate=None, moe_cache_auto=False, kv_reserve_tokens=16384, moe_cache_policy='lru', moe_prefill_overlap=True, moe_prefill_hit_d2d=False, moe_collect_stats=False, moe_cpu_threads=0, moe_cpu_layers=None, moe_hybrid_max_fetch=-1, cuda_graph_bs=None, cuda_graph_max_bs=None, page_size=1, memory_ratio=0.9, linear_state_cache_ratio=2.0, swa_full_tokens_ratio=0.2, swa_num_pages_override=None, distributed_timeout=60.0, use_dummy_weight=False, use_pynccl=True, max_seq_len_override=16384, num_page_override=None, num_token_override=None, mm=MultimodalConfig(disabled_encoders=frozenset(), embed_cache_device='cpu', encoder_weights='host', image_min_tokens=None, image_max_tokens=None, processor_kwargs={}), max_exte

In [ ]:
def chat(prompt, max_tokens=800, effort="none", timeout=600, **kw):
    if not alive():
        raise RuntimeError("ft serve is not running.\n\n" + diagnose())
    body = {"model": SERVED, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "stream": False, **kw}
    if effort:
        body["reasoning_effort"] = effort
    r = requests.post(f"{BASE}/v1/chat/completions", json=body, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}\n\n" + diagnose())
    msg = r.json()["choices"][0]["message"]

    if not msg.get("content"):
        think = (msg.get("reasoning_content") or "")[-500:]

        raise RuntimeError(
            f"empty content — the model used all {max_tokens} tokens thinking.\n"
            f"Raise max_tokens or use effort='none'.\n\n...{think}")

    return msg["content"]

print(chat("Explain Mixture of Experts in two sentences."))

Mixture of Experts is a machine learning architecture that splits a task among multiple specialized sub-models, known as "experts." A gating network dynamically routes inputs to the most suitable experts, allowing the system to leverage specialized knowledge for more efficient and accurate predictions than a single monolithic model.


In [ ]:
assert alive(), "server is down — don't expose a dead origin"

CF_LOG = "/content/cloudflared.log"
for pid in subprocess.run(["pgrep", "-f", "cloudflared tunnel"],
                          capture_output=True, text=True).stdout.split():
    try: os.kill(int(pid), signal.SIGTERM)
    except Exception: pass

subprocess.Popen(["cloudflared", "tunnel", "--url", BASE],
                 stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)

PUBLIC_URL = None
for _ in range(40):
    time.sleep(2)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CF_LOG).read())
    if found: PUBLIC_URL = found[-1]; break

if not PUBLIC_URL:
    print(open(CF_LOG).read()[-1500:])
else:
    print(PUBLIC_URL)
    print("models :", f"{PUBLIC_URL}/v1/models")
    print("chat   :", f"{PUBLIC_URL}/v1/chat/completions")

    status = None
    for i in range(20):
        try:
            status = requests.get(f"{PUBLIC_URL}/v1/models", timeout=30).status_code
            print(f"status : {status} (after {i*5}s)")
            break
        except requests.exceptions.ConnectionError as e:
            if "NameResolution" not in str(e) and "getaddrinfo" not in str(e):
                print("status : connection error —", str(e)[:120]); break
            print(f"  waiting for DNS... {i*5}s", end="\r")
            time.sleep(5)
    else:
        print("status : DNS not resolving from Colab yet — the URL still works "
              "from outside, test it in your browser.")

https://multimedia-potatoes-therapist-singing.trycloudflare.com
models : https://multimedia-potatoes-therapist-singing.trycloudflare.com/v1/models
chat   : https://multimedia-potatoes-therapist-singing.trycloudflare.com/v1/chat/completions
status : 200 (after 30s)


In [ ]:
!pip -q install openai
from openai import OpenAI

client = OpenAI(api_key="dummy", base_url=f"{PUBLIC_URL}/v1")

r = client.chat.completions.create(
    model=SERVED,
    messages=[{"role": "user", "content": "Give me 3 facts about Mixture of Experts."}],
    max_tokens=800,                      # thinking eats the budget first
    reasoning_effort="none",             # 'none' | 'minimal' | 'low' | 'medium' | 'high' | 'max'
    timeout=600)

m = r.choices[0].message
print(m.content or "[no content — still thinking]\n\n" + (m.reasoning_content or "")[:2000])
print("\nusage:", r.usage)

Here are three key facts about the **Mixture of Experts (MoE)** architecture:

1. **Conditional Computation and Sparse Activation**: Unlike dense neural networks where all parameters are used for every input, MoE models use a *gating network* to dynamically route each input to a small subset of specialized “expert” networks. Typically, only 2–8 out of hundreds or thousands of experts are activated per token, making inference significantly more energy-efficient and allowing for much larger total model capacity without proportional increases in compute cost.

2. **Discovered Expert Specialization**: During training, experts tend to naturally specialize in different aspects of the data. For example, in language models, some experts may become adept at handling mathematical reasoning, others at coding, and others at creative writing or specific languages—without being explicitly programmed to do so. This emergent specialization contributes to the model’s overall performance and versatility

In [ ]:
import os

os.environ["PUBLIC_URL"] = PUBLIC_URL

In [ ]:
!curl "$PUBLIC_URL/v1/chat/completions" \
  -H 'Content-Type: application/json' \
  -d '{"model":"qwen","messages":[{"role":"user","content":"hello"}],"max_tokens":800,"reasoning_effort":"none"}'

{"id":"chatcmpl-11","object":"chat.completion","created":1789849556,"model":"qwen","choices":[{"index":0,"message":{"role":"assistant","content":"Hello! How can I help you today?"},"finish_reason":"stop"}],"usage":{"prompt_tokens":13,"completion_tokens":10,"total_tokens":23}}

In [ ]:
# stop_server()
# for pid in subprocess.run(["pgrep", "-f", "cloudflared tunnel"],
#                           capture_output=True, text=True).stdout.split():
#     try: os.kill(int(pid), signal.SIGTERM)
#     except Exception: pass
# print("stopped. re-run cell 4 to start again.")